# STITCHV2 Model Deep Dive

This notebook explains the STITCHV2 model and execution pipeline in detail. It is documentation, not a benchmark result file. Keep measured results in benchmark output directories and markdown reports generated by the benchmark scripts.

Primary code references:

- `src/stitchv2/config.py`: public configuration surface
- `src/stitchv2/pipeline.py`: orchestration, IO, ploidy grouping, Dask execution, output writing
- `src/stitchv2/hmm.py`: HMM kernels, JAX paths, EM founder updates
- `src/stitchv2/calibration.py`: posterior calibration, masked CV, sanity guards
- `src/stitchv2/pedigree.py`: kinship and transmission post-processing
- `src/stitchv2/output.py`: Parquet combine, optional float Zarr writing, explicit BCF interoperability export
- `benchmarks/generate_synthetic_stitch_stitchv2_benchmark.py`: synthetic STITCH/STITCHV2 comparison generator
- `benchmarks/generate_mouse_stitch_stitchv2_benchmark.py`: original mouse-data comparison generator


In [ ]:
from pathlib import Path
import sys

REPO = Path('/home/bonnie/Documents/codex/STITCHV2')
sys.path.insert(0, str(REPO / 'src'))
print(REPO)


## 1. Model Goal

STITCHV2 imputes genotypes from low-coverage sequencing reads. The model treats each sample genome as a mosaic of `K` founder or ancestral haplotypes. At each variant, a sample has one or more hidden founder states, depending on ploidy. Reads provide noisy evidence for reference or alternate alleles, and the HMM smooths this evidence along the chromosome using generation-scaled recombination probabilities.

The output surface is intentionally table-first:

- Hard calls and categorical fields stay in Parquet.
- Dosage, genotype posterior, transition, recombination, and founder-probability values can be combined as lazy xarray/Dask arrays and optionally written to Zarr.
- VCF export is intentionally disabled. BCF exists only as a slow interoperability path for external tools.

## 2. Inputs

### Samples

`samples.parquet` or a delimited text table must include `generation`; `sample_id` and `bam_path` are normalized if absent. Optional fields include `sex` for sex-chromosome ploidy and `plink_path` for per-sample hard-call evidence.

### Positions

`positions.parquet` or text table contains `CHR`, `POS`, `REF`, `ALT`. Positions are filtered by `--chromosome`, `--chr-start`, and `--chr-end`, then split into blocks.

### Founders

STITCHV2 supports uniform mutable founders, VCF founders, PLINK founders, and in-memory `FounderPanel`. For STITCH-parity fixed-founder comparisons, use:

```bash
--founder-vcf founders.truth.vcf.gz --founder-immutable --fragment-coupling-model stitch_parity
```

When no founder file is supplied, mutable founder probabilities are learned by EM. These no-founder runs should use enough iterations to converge; the benchmark generators default to `--iterations 40` for this reason.

## 3. Hidden State Space

For diploid samples, the hidden state at variant `t` is a founder pair `(a, b)`. There are `K * K` ordered pair states in the fast diploid implementation. For pseudo-haploid samples, the state is a single founder. For polyploid samples, STITCHV2 can use a generic ploidy state representation.

Ploidy-zero samples are a special case: they are retained in output, but skipped by the HMM. Their genotype calls are missing, dosage/posterior outputs are missing where appropriate, and haplotype-probability outputs are zero vectors.

## 4. Transition Model

The transition step approximates a Li-Stephens-style mosaic process. Recombination probability increases with genetic distance and sample generation. Adjacent variants with small genetic distance should usually keep the same founder state; variants separated by more recombination opportunity switch more often.

Important parameters:

- `generation`: per-sample mosaic age.
- `--n-founders`: size of the founder state space.
- `--block-size`: number of variants per independent HMM block.
- `--write-transitions`: writes compact transition summaries for diagnostics.

Transition diagnostics are useful for debugging underfitting, unexpected switch rates, and chromosome-window boundary behavior.

## 5. Read Emission Model

For each sample and variant, read evidence is summarized into reference, alternate, and other observations. Given founder alternate probabilities, STITCHV2 computes the likelihood of the observed read counts under each hidden state.

The key error terms are:

- `--sequencing-error-rate` through `HMMConfig.sequencing_error_rate`.
- Per-read quality values when available.
- `--fragment-max-diff-reads` and `--fragment-max-emission-diff` numerical guardrails.

The fast JAX count-emission path keeps count-to-emission work on device for common diploid and pseudo-haploid cases.

## 6. Fragment Coupling

Read fragments can cover multiple SNPs. STITCHV2's benchmark/parity mode uses:

```bash
--fragment-coupling-model stitch_parity --fragment-likelihood-mode augment
```

In this mode, multi-SNP read evidence contributes coupled likelihood terms. The JAX fragment path builds fragment haplotype likelihoods with `segment_sum` and then injects those likelihoods into diploid emissions. This avoids Python loops over fragments in the core HMM path.

The fragment likelihood path rescales large log-likelihood differences for numerical stability. The implementation uses a conditional branch so the unscaled exponential is not evaluated when rescaling is enabled.

## 7. Forward-Backward and Outputs

The HMM computes posterior probabilities over hidden states. From those posteriors STITCHV2 derives:

- `dosage`: expected alternate allele count
- `genotype_posterior`: probability of each alternate allele count
- `genotype_call`: hard call or `-1` no-call
- `haplotype_probabilities`: founder/haplotype posterior summaries when requested
- `recombination` and `transitions`: diagnostics
- `founder_updates`: mutable-founder EM results

For diploid GP, genotype classes are usually `[0, 1, 2]`. For ploidy `P`, there are `P + 1` dosage classes.

## 8. EM Founder Updates

When founders are mutable, STITCHV2 updates founder alternate probabilities from posterior expected allele assignments. Immutable founders are never changed. Damping can stabilize difficult no-founder runs:

```bash
--em-founder-update-damping 0.5
```

Convergence controls:

- `--em-iterations`: maximum EM passes
- `--em-convergence-tol`: stop threshold; use `0` to force the requested number of iterations
- `--em-convergence-min-iterations`
- `--em-convergence-patience`

For strict benchmark comparability with no founders, the new synthetic benchmark generator defaults to 40 iterations and disables early stopping.

## 9. Calibration

Calibration adjusts genotype posterior probabilities after the HMM. STITCHV2 supports fixed temperature/blend calibration, masked-CV grid selection, and optional LightGBM/isotonic calibration when labels are available.

Calibration sanity guards compare raw and calibrated posteriors using dosage shift, MAF shift, heterozygosity shift, and entropy shift. If calibration moves the posterior too aggressively, the guard can fall back to raw HMM posteriors.

Useful outputs for inspecting calibration:

- `stage_timings.json`: per-block calibration time and memory
- `genotype_posteriors/*.parquet`: raw/calibrated GP depending on run mode
- Benchmark violin plots for per-variant R2, accuracy, balanced accuracy, F1, info, MAF, HWE, and missingness

## 10. Pedigree Modes

Pedigree processing happens after the population HMM. It does not replace read evidence; it smooths or refines posterior summaries using family relationships.

Modes:

- `off`: no pedigree post-processing
- `smooth`: dosage smoothing
- `kinship`: kinship-aware fallback and smoothing
- `transmission`: parent-offspring message passing

Pedigree validation should be benchmarked separately from founder-mode validation because it changes the posterior after the HMM.

## 11. IO Model

STITCHV2 is Parquet/Zarr-first.

- Hard calls and categorical fields use Parquet. Dictionary encoding, RLE, bit-packing, and compact integer types are a good fit.
- Dosages, genotype posteriors, transition probabilities, recombination rates, and founder probabilities can be represented as lazy xarray/Dask arrays and written to Zarr.
- BCF export is an explicit slow interoperability path.
- VCF export is disabled.

CLI example:

```bash
stitchv2 combine --run-output-dir runs/chr1 --write-zarr
```

## 12. Dask and Reliability

Dask schedules coarse HMM leaf tasks over blocks and sample batches. It should not be used to split individual dynamic-programming recurrences into tiny tasks. Good defaults are a moderate number of workers and coarse sample batches.

Recommended local defaults:

```bash
--executor dask \
--dask-scheduler local \
--dask-n-workers 4 \
--dask-threads-per-worker 1 \
--dask-sample-batch-size 0
```

Use `--dask-scheduler threads` when socket/dashboard behavior is unstable. Keep `--dask-processes` off by default to avoid extra serialization cost for JAX arrays and founder payloads.

## 13. Benchmark Generators

Synthetic default benchmark:

```bash
python benchmarks/generate_synthetic_stitch_stitchv2_benchmark.py \
  --run-root benchmark_runs/synthetic_stitch_stitchv2_review \
  --stitchv2 stitchv2 \
  --n-samples 1000 \
  --n-variants 2000 \
  --coverage 0.1 \
  --iterations 40
```

Original STITCH mouse-data benchmark:

```bash
python benchmarks/generate_mouse_stitch_stitchv2_benchmark.py \
  --run-root benchmark_runs/mouse_stitch_stitchv2_review \
  --data-dir benchmark_runs/STITCH_example_2016_05_10_data \
  --chromosome chr19 \
  --iterations 40
```

Both generators write `stitch_stitchv2_comparison.md`, runtime/memory plots, stage stacked bars, ROC curves, and per-variant quality/QC violin plots.

## 14. Improvement Roadmap

High-impact speed and memory items:

1. Add shape-bucketed persistent JAX compilation cache defaults in benchmark scripts and production presets.
2. Keep founder panels and immutable per-block config scattered in Dask workers; avoid repeatedly serializing large arrays.
3. Add a full haploid-with-fragments JAX kernel so sex-chromosome fragment evidence does not fall back to NumPy assembly.
4. Add GPU-oriented task placement: one Dask worker per GPU, preallocated device buffers, and fewer host/device transfers.
5. Use adaptive EM stopping for production, but force fixed iterations for benchmarks.
6. Add calibration training-set stratification by MAF, depth, missingness, and HWE deviation to improve rare-variant behavior.
7. Add posterior uncertainty diagnostics that flag all-heterozygote collapse, low-information sites, or overconfident monomorphic calls.
8. Add dask-jobqueue runner templates once local Dask is stable.
9. Add chunk-size autotuning based on observed reads per block, not only sample/variant counts.
10. Add benchmark gates that fail if STITCHV2 loses fixed-founder parity, ploidy-zero output behavior, or calibration sanity behavior.